# Mapa 3D de las estaciones del Gran Concepción

Genera un mapa navegable en 3D con relieve, las comunas coloreadas y las 16 estaciones de monitoreo.

- **Salida única:** `data/mapa_3d_estaciones.html` (los íconos van dentro del archivo).
- **Requiere** en `src/config.py`: `ESTACIONES`, `UTM_ESTACIONES` (coordenadas corregidas) y `RED_ESTACIONES`.
- **Librerías del mapa:** MapLibre GL JS, mapa base OpenFreeMap, relieve AWS Terrain Tiles. Todas gratuitas y sin clave.
- Se necesita internet al abrir el HTML (carga el mapa base y el relieve).

In [ ]:
import sys, json
sys.path.append("..")
from src.config import ESTACIONES, UTM_ESTACIONES, RED_ESTACIONES
from src.meteo import utm_a_latlon

## 1. Configuración: sub-zonas y colores

Para cambiar un color, edítalo solo aquí: se aplica a la comuna y a sus estaciones.

In [ ]:
SUBZONAS = {
    "Coronel": (["831", "878", "880", "881", "879"], "#e65b24"),
    "Lota": (["859", "860"], "#8b5cf6"),
    "Talcahuano": (["802", "807", "806"], "#2a78d6"),
    "Hualpen": (["805", "838", "804"], "#14b8c4"),
    "Concepcion": (["827"], "#e6155e"),
    "Chiguayante": (["854"], "#f2b739"),
    "Hualqui": (["841"], "#0bde75"),
}

# Nombre de la sub-zona -> nombre de la comuna como viene en el mapa (con tilde)
COMUNA_DE_SZ = {"Coronel": "Coronel", "Lota": "Lota", "Talcahuano": "Talcahuano",
                "Hualpen": "Hualpén", "Concepcion": "Concepción",
                "Chiguayante": "Chiguayante", "Hualqui": "Hualqui"}
SIN_ESTACION = ["San Pedro de la Paz", "Penco", "Tomé"]   # zona saturada, sin datos

SALIDA = "../data/mapa_3d_estaciones.html"

## 2. Colores de las comunas y datos de las estaciones

In [ ]:
# Color de cada comuna: el de su sub-zona; las sin estación en gris
COLOR_COMUNA = {COMUNA_DE_SZ[sz]: col for sz, (_, col) in SUBZONAS.items()}
COLOR_COMUNA.update({c: "#9e9e9e" for c in SIN_ESTACION})

expr_color = ["match", ["get", "Comuna"]]
for comuna, color in COLOR_COMUNA.items():
    expr_color += [comuna, color]
expr_color.append("rgba(0,0,0,0)")   # el resto de la región, transparente

# Red de cada estación, desde config.py
PUBLICAS = {c for c, (red, _) in RED_ESTACIONES.items() if red == "publica"}

coords = utm_a_latlon(UTM_ESTACIONES)
sz_de = {c: sz for sz, (cods, _) in SUBZONAS.items() for c in cods}
color_de = {c: col for cods, col in SUBZONAS.values() for c in cods}

estaciones = [{
    "nombre": ESTACIONES[c][0], "subzona": sz_de[c], "color": color_de[c],
    "red": "Pública (MMA)" if c in PUBLICAS else "Privada",
    "operador": RED_ESTACIONES[c][1],
    "lat": coords[c][0], "lon": coords[c][1],
} for c in ESTACIONES]
estaciones.sort(key=lambda e: e["lat"])   # recorrido de sur a norte

print(f"{len(estaciones)} estaciones ({len(PUBLICAS)} públicas)")

## 3. Plantilla del mapa (HTML + JavaScript)

No hace falta tocar esta celda. `__DATOS__` y `__COLORES__` se reemplazan con los datos de la celda anterior.

In [ ]:
html = """<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>Estaciones Gran Concepción 3D</title>
<link href="https://unpkg.com/maplibre-gl@4.7.1/dist/maplibre-gl.css" rel="stylesheet">
<script src="https://unpkg.com/maplibre-gl@4.7.1/dist/maplibre-gl.js"></script>
<style>
  html, body, #map { margin: 0; height: 100%; font-family: system-ui, sans-serif; }
  .est { width: 32px; height: 42px; cursor: pointer; filter: drop-shadow(0 2px 3px #0007); }
  .est svg, .punto svg { display: block; }
  .punto { flex: none; }
  #panel { position: absolute; top: 10px; left: 10px; width: 230px; max-height: 90%;
           overflow-y: auto; background: #fffe; border-radius: 8px; padding: 10px;
           box-shadow: 0 2px 8px #0004; font-size: 13px; }
  #panel h3 { margin: 0 0 8px; font-size: 15px; }
  #panel button.ctrl { width: 49%; padding: 6px; margin-bottom: 8px; border: 0;
           border-radius: 5px; background: #2a78d6; color: white; cursor: pointer; }
  #panel button.ctrl.activo { background: #e65b24; }
  .item { display: flex; align-items: center; gap: 6px; padding: 4px; border-radius: 4px; cursor: pointer; }
  .item:hover { background: #eee; }
  #nota { color: #666; font-size: 11px; margin-top: 6px; line-height: 1.6; }
  #nota svg { display: inline-block; vertical-align: middle; }
</style></head>
<body><div id="map"></div>
<div id="panel">
  <h3>Estaciones Gran Concepción</h3>
  <button class="ctrl" id="btnGirar">Girar</button>
  <button class="ctrl" id="btnTour">Recorrido</button>
  <div id="lista"></div>
  <div id="nota"></div>
</div>
<script>
const ESTACIONES = __DATOS__;
const VISTA_GENERAL = { center: [-73.08, -36.88], zoom: 10.6, pitch: 60, bearing: -20 };

// Pin con ícono: mástil de monitoreo (pública) o fábrica (privada)
function icono(color, privada, ancho = 32) {
  const alto = ancho * 42 / 32;
  const dibujo = privada
    ? `<path d="M9.5 21.5V14l3.5 2.6V14l3.5 2.6V14l3.5 2.6V9.5h2.5v12z" fill="${color}"/>`
    : `<g stroke="${color}" stroke-width="1.8" fill="none" stroke-linecap="round">
         <path d="M16 11.5L12.5 22M16 11.5L19.5 22M13.6 18.5h4.8"/>
         <path d="M12.6 7.6q-1.6 2.4 0 4.8M19.4 7.6q1.6 2.4 0 4.8"/>
       </g><circle cx="16" cy="10" r="1.7" fill="${color}"/>`;
  return `<svg width="${ancho}" height="${alto}" viewBox="0 0 32 42">
    <path d="M16 1C8.3 1 2 7.3 2 15c0 10.2 14 26 14 26s14-15.8 14-26C30 7.3 23.7 1 16 1z"
          fill="${color}" stroke="white" stroke-width="2"/>
    <circle cx="16" cy="15" r="9.5" fill="white"/>${dibujo}</svg>`;
}

// Leyenda del panel
nota.innerHTML = `${icono("#555", false, 14)} pública (MMA) &nbsp; ${icono("#555", true, 14)} privada (empresa)
  <br>Gris: comuna de la zona saturada sin estación<br>Clic derecho + arrastrar para girar e inclinar`;

const map = new maplibregl.Map({
  container: "map", style: "https://tiles.openfreemap.org/styles/liberty",
  ...VISTA_GENERAL, maxPitch: 85
});
map.addControl(new maplibregl.NavigationControl({ visualizePitch: true }));
map.addControl(new maplibregl.FullscreenControl());

// ---------- Movimiento ----------
let girando = false, enTour = false;

function girar() {
  if (!girando) return;
  map.setBearing(map.getBearing() + 0.08);
  requestAnimationFrame(girar);
}
function detenerTodo() {
  girando = false; enTour = false;
  btnGirar.classList.remove("activo"); btnTour.classList.remove("activo");
  btnTour.textContent = "Recorrido";
}
function volarA(e) {
  return new Promise(ok => {
    map.flyTo({ center: [e.lon, e.lat], zoom: 14.3, pitch: 70,
                bearing: map.getBearing() + 60, duration: 4000, essential: true });
    map.once("moveend", ok);
  });
}
const esperar = ms => new Promise(ok => setTimeout(ok, ms));

async function recorrido() {
  for (const e of ESTACIONES) {
    if (!enTour) return;
    await volarA(e);
    e.popup.addTo(map);
    await esperar(1800);
    e.popup.remove();
  }
  if (enTour) map.flyTo({ ...VISTA_GENERAL, duration: 4000 });
  detenerTodo();
}

btnGirar.onclick = () => {
  const activar = !girando; detenerTodo();
  if (activar) { girando = true; btnGirar.classList.add("activo"); girar(); }
};
btnTour.onclick = () => {
  const activar = !enTour; detenerTodo();
  if (activar) { enTour = true; btnTour.classList.add("activo");
                 btnTour.textContent = "Detener"; recorrido(); }
};
// Cualquier interacción con el mapa detiene el giro automático
["mousedown", "touchstart", "wheel"].forEach(ev =>
  map.getCanvas().addEventListener(ev, () => { if (girando) detenerTodo(); }));

// ---------- Capas ----------
map.on("load", () => {
  map.addSource("relieve", {
    type: "raster-dem", encoding: "terrarium", tileSize: 256, maxzoom: 14,
    tiles: ["https://s3.amazonaws.com/elevation-tiles-prod/terrarium/{z}/{x}/{y}.png"]
  });
  map.setTerrain({ source: "relieve", exaggeration: 1.8 });

  const primerSimbolo = map.getStyle().layers.find(l => l.type === "symbol").id;
  map.addLayer({ id: "sombra", type: "hillshade", source: "relieve",
                 paint: { "hillshade-exaggeration": 0.5 } }, primerSimbolo);

  map.addSource("comunas", { type: "geojson",
    data: "https://raw.githubusercontent.com/caracena/chile-geojson/master/8.geojson" });

  // Relleno de cada comuna con su color (debajo de los nombres del mapa)
  map.addLayer({ id: "relleno", type: "fill", source: "comunas",
    paint: { "fill-color": __COLORES__, "fill-opacity": 0.30 } }, primerSimbolo);

  map.addLayer({ id: "limites", type: "line", source: "comunas",
    filter: ["in", ["get", "Comuna"], ["literal",
      ["Coronel","Lota","Talcahuano","Hualpén","Concepción","Chiguayante","Hualqui",
       "San Pedro de la Paz","Penco","Tomé"]]],
    paint: { "line-color": "#333", "line-width": 1.5, "line-dasharray": [2, 1] } });

  // Estaciones: pin en el mapa + fila en el panel
  for (const e of ESTACIONES) {
    const privada = e.red === "Privada";
    e.popup = new maplibregl.Popup({ offset: 40, closeButton: false })
      .setLngLat([e.lon, e.lat])
      .setHTML(`<b>${e.nombre}</b><br>${e.subzona}<br>${privada ? "Privada · " + e.operador : e.red}`);

    const el = document.createElement("div");
    el.className = "est";
    el.innerHTML = icono(e.color, privada);
    new maplibregl.Marker({ element: el, anchor: "bottom" })   // la punta del pin marca la ubicación
      .setLngLat([e.lon, e.lat]).setPopup(e.popup).addTo(map);

    const fila = document.createElement("div");
    fila.className = "item";
    fila.innerHTML = `<span class="punto">${icono(e.color, privada, 14)}</span>${e.nombre}
                      <span style="color:#888;margin-left:auto">${e.subzona}</span>`;
    fila.onclick = async () => { detenerTodo(); await volarA(e); e.popup.addTo(map); };
    lista.appendChild(fila);
  }
});
</script></body></html>"""

## 4. Generar el archivo

In [ ]:
with open(SALIDA, "w", encoding="utf-8") as f:
    f.write(html.replace("__DATOS__", json.dumps(estaciones, ensure_ascii=False))
                .replace("__COLORES__", json.dumps(expr_color, ensure_ascii=False)))
print(f"Listo: {SALIDA} (ábrelo con doble clic en el navegador)")

## 5. (Opcional) Verlo dentro del notebook

Si VS Code no lo muestra, ábrelo directamente en el navegador. Ahí funciona mejor: pantalla completa y giro con clic derecho.

In [ ]:
from IPython.display import IFrame
IFrame(SALIDA, width="100%", height=650)